# OnePilot Sprint 10 Benchmark
## Agent Precision : schema structure vs ReAct+ hallucination

Objectifs : mesurer le taux de succes, les hallucinations, et la latence.


In [ ]:
import requests
import time
import re
import json
import unicodedata
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

BASE_URL  = 'http://onepilot_api:8000'
SOURCE_ID = '03add1dc-754a-476a-bbd5-1e53a05bf8d7'

try:
    r = requests.get(f'{BASE_URL}/health', timeout=5)
    print(f'API connectee status={r.status_code}')
except Exception as e:
    print(f'Connexion echouee : {e}')


In [ ]:
HALLUCINATION_TABLES = [
    'ligne_de_financement', 'di_finline_dtls', 'di_fnline', 'di_trnb',
    'di_tl_prppmt', 'ligne_financement', 'financement_table',
    'gs_blngst_comp', 'rc_b2gsc_2_bnksttp',
]

def _norm(s):
    import unicodedata as ud
    s = ud.normalize('NFD', s.lower())
    return ''.join(c for c in s if ud.category(c) != 'Mn')

def detect_hallucination(sql):
    if not sql: return True, ['SQL vide']
    sql_norm = _norm(sql)
    found = [t for t in HALLUCINATION_TABLES if _norm(t) in sql_norm]
    return len(found) > 0, found

def validate_coverage(question, sql):
    if not sql: return False, 0.0, ['SQL vide']
    entities = {}
    years = re.findall(r'\\b(?:19|20)\\d{2}\\b', question)
    if years: entities['years'] = years
    banks = ['bnp', 'banque postale', 'societe generale']
    for b in banks:
        if b in _norm(question): entities.setdefault('banks', []).append(b)
    if not entities: return True, 1.0, []
    sql_norm = _norm(sql)
    missing, total, covered = [], 0, 0
    for etype, vals in entities.items():
        for v in vals:
            total += 1
            if _norm(v) in sql_norm: covered += 1
            else: missing.append(f'{etype}:{v}')
    score = covered / total if total > 0 else 1.0
    return score >= 0.6, round(score, 2), missing

def query_s10(question, verbose=True):
    t0 = time.time()
    try:
        r = requests.post(f'{BASE_URL}/orchestrator/query',
            json={'question': question, 'source_id': SOURCE_ID, 'dialect': 'mssql'},
            timeout=180)
        elapsed = round((time.time() - t0) * 1000)
        data  = r.json()
        sql   = data.get('sql', '')
        agent = data.get('agent_type', 'unknown')
        iters = data.get('iterations', 1)
        is_hall, hall_t = detect_hallucination(sql)
        cov_ok, cov_sc, missing = validate_coverage(question, sql)
        success = bool(sql and not is_hall and cov_ok)
        if verbose:
            icons = {'direct_sql':'Direct   ','precision':'Precision','react_plus':'ReAct+   ','multi_query':'Multi    '}
            tag = icons.get(agent, agent[:9])
            ok  = 'OK  ' if success else 'FAIL'
            hall_s = f' HALL:{hall_t[:1]}' if is_hall else ''
            cov_s  = f' cov={cov_sc:.0%}' if missing else ' cov=OK'
            print(f'  [{ok}] [{tag}] {elapsed:>6}ms{cov_s}{hall_s}  {question[:55]}')
        return {'question':question,'sql':sql,'agent':agent,'iterations':iters,
                'elapsed_ms':elapsed,'success':success,'cov_score':cov_sc,
                'missing':missing,'is_hallucination':is_hall,'halluc_tables':hall_t}
    except Exception as e:
        if verbose: print(f'  ERREUR : {e}')
        return {'question':question,'sql':'','agent':'error','iterations':0,
                'elapsed_ms':0,'success':False,'cov_score':0.0,'missing':[str(e)],
                'is_hallucination':True,'halluc_tables':[str(e)]}

print('Fonctions OK')


In [ ]:
# TIER 1 : Direct SQL
print('='*65)
print('  TIER 1 - Direct SQL (objectif 0 hallucination)')
print('='*65)

TIER1 = [
    'liste les devises',
    'financements actifs',
    'solde bancaire',
    'comptes BNP',
    'comptes Societe Generale',
    'utilisateurs bloques',
    'taux de change',
    'financements clotures',
    'financements BNP',
    'transactions TND superieures a 10000 La Banque Postale 2024',
]

results_t1 = [query_s10(q) for q in TIER1]
ok_t1   = [r for r in results_t1 if r['success']]
hall_t1 = [r for r in results_t1 if r['is_hallucination']]
print(f'\n  Succes={len(ok_t1)}/{len(results_t1)} | Hall={len(hall_t1)} | Moy={round(sum(r["elapsed_ms"] for r in results_t1)/len(results_t1))}ms')


In [ ]:
# TIER 2 : Aggregations complexes (Agent Precision)
print('='*65)
print('  TIER 2 - Aggregations complexes (Agent Precision)')
print('='*65)

TIER2 = [
    'total des financements par banque et par devise en 2024',
    'top 10 financements par banque et par type de transaction',
    'financements par banque et par societe',
    'total des financements par type de transaction en 2023',
    'repartition des financements par banque et par type',
    'financements par banque et par devise en 2025',
    'total des financements par type et par etat',
    'nombre de financements par banque et par societe',
]

results_t2 = [query_s10(q) for q in TIER2]
ok_t2   = [r for r in results_t2 if r['success']]
hall_t2 = [r for r in results_t2 if r['is_hallucination']]
print(f'\n  Succes={len(ok_t2)}/{len(results_t2)} | Hall={len(hall_t2)} | Moy={round(sum(r["elapsed_ms"] for r in results_t2)/len(results_t2))}ms')


In [ ]:
# TIER 3 : Sous-requetes et analyses avancees
print('='*65)
print('  TIER 3 - Sous-requetes et analyses avancees')
print('='*65)

TIER3 = [
    'financements dont la maturite depasse la duree moyenne des financements BNP',
    'financements dont le montant est superieur a la moyenne',
    'top 5 banques par montant total de financements actifs',
    'comparaison montant moyen des financements par type de transaction',
    'financements ouverts dont la maturite depasse 1000 jours',
]

results_t3 = [query_s10(q) for q in TIER3]
ok_t3   = [r for r in results_t3 if r['success']]
hall_t3 = [r for r in results_t3 if r['is_hallucination']]
print(f'\n  Succes={len(ok_t3)}/{len(results_t3)} | Hall={len(hall_t3)} | Moy={round(sum(r["elapsed_ms"] for r in results_t3)/len(results_t3))}ms')


In [ ]:
# RESUME FINAL
print('='*65)
print('  RESUME Sprint 10')
print('='*65)

all_r    = results_t1 + results_t2 + results_t3
all_ok   = [r for r in all_r if r['success']]
all_hall = [r for r in all_r if r['is_hallucination']]
all_dir  = [r for r in all_r if 'direct' in r.get('agent','')]
all_prec = [r for r in all_r if r.get('agent') == 'precision']
all_rea  = [r for r in all_r if r.get('agent') == 'react_plus']

print(f'\n  Total questions : {len(all_r)}')
print(f'  Succes         : {len(all_ok)}/{len(all_r)} ({len(all_ok)/len(all_r):.0%})')
print(f'  Hallucinations : {len(all_hall)}/{len(all_r)} ({len(all_hall)/len(all_r):.0%})')
print(f'\n  Par agent :')
print(f'    Direct SQL : {len(all_dir)}')
print(f'    Precision  : {len(all_prec)}')
print(f'    ReAct+     : {len(all_rea)}')
avg = lambda l: round(sum(r['elapsed_ms'] for r in l)/len(l)) if l else 0
print(f'\n  Latence :')
print(f'    T1 Direct   : {avg(results_t1)}ms')
print(f'    T2 Complexe : {avg(results_t2)}ms')
print(f'    T3 Avance   : {avg(results_t3)}ms')
hall_rate = len(all_hall)/len(all_r)
t2_rate = len(ok_t2)/len(results_t2)
print(f'\n  Objectifs Sprint 10 :')
print(f'    Hallucinations < 10% : {"OK" if hall_rate < 0.1 else "FAIL"} ({hall_rate:.0%})')
print(f'    Succes T2 > 70%      : {"OK" if t2_rate > 0.7 else "FAIL"} ({t2_rate:.0%})')


In [ ]:
# GRAPHIQUES
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('OnePilot Sprint 10 - Agent Precision Benchmark', fontsize=14, fontweight='bold')

tiers_ok    = [len(ok_t1), len(ok_t2), len(ok_t3)]
tiers_total = [len(results_t1), len(results_t2), len(results_t3)]
tiers_fail  = [t-ok for t,ok in zip(tiers_total, tiers_ok)]
x = np.arange(3)
axes[0].bar(x, tiers_ok,   label='Succes', color='#2ecc71', alpha=0.85)
axes[0].bar(x, tiers_fail, bottom=tiers_ok, label='Echec', color='#e74c3c', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(['T1 Direct','T2 Complexe','T3 Avance'])
axes[0].set_title('Succes par tier'); axes[0].set_ylabel('Questions'); axes[0].legend()
for i,(ok,t) in enumerate(zip(tiers_ok,tiers_total)):
    axes[0].text(i, t+0.1, f'{ok}/{t}', ha='center', fontweight='bold')

avg_t = lambda l: round(sum(r['elapsed_ms'] for r in l)/len(l)) if l else 0
latences = [avg_t(results_t1), avg_t(results_t2), avg_t(results_t3)]
bars = axes[1].bar(['T1','T2','T3'], latences, color=['#2ecc71','#9b59b6','#e67e22'], alpha=0.85, width=0.5)
axes[1].axhline(500, color='green', linestyle='--', alpha=0.5, label='500ms')
axes[1].axhline(5000, color='orange', linestyle='--', alpha=0.5, label='5s')
axes[1].set_title('Latence par tier'); axes[1].set_ylabel('ms'); axes[1].legend(fontsize=8)
for bar,v in zip(bars,latences):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50, f'{v}ms', ha='center', fontsize=10, fontweight='bold')

hall_c = [len([r for r in results_t1 if r['is_hallucination']]),
          len([r for r in results_t2 if r['is_hallucination']]),
          len([r for r in results_t3 if r['is_hallucination']])]
bars3 = axes[2].bar(['T1','T2','T3'], hall_c, color=['#2ecc71','#9b59b6','#e67e22'], alpha=0.85, width=0.5)
axes[2].set_title('Hallucinations par tier'); axes[2].set_ylabel('Nb tables inventees')
for bar,v in zip(bars3,hall_c):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, str(v), ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('benchmark_sprint10.png', dpi=150, bbox_inches='tight')
plt.close()
print('Graphique sauvegarde : benchmark_sprint10.png')
